In [53]:
import findspark
# Automatically locate the Spark installation
findspark.init() 

# Now you can safely import pyspark
import pyspark
print(pyspark.__version__)


4.2.0


In [54]:
import os
import findspark

# Point to your newly configured Java and Spark paths
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
os.environ["SPARK_HOME"] = r"C:\Program Files\Spark\spark-4.2.0-bin-hadoop3\spark-4.2.0-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\Program Files\hadoop"
os.environ["PATH"] = r"C:\Program Files\hadoop\bin;" + os.environ.get("PATH", "")


# 2. Append the bin folder directly to your running Python session path
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"
# Initialize findspark with your exact directory
findspark.init(os.environ["SPARK_HOME"])
 


In [55]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailSalesAnalysis")
    .getOrCreate()
)
df = spark.read.csv(
    "D:/Portfolio/Retail Sales Dataset/retail-sales-analysis/Sales.csv",
    header=True,
    inferSchema=True
)

In [56]:
df.printSchema()


root
 |-- Transaction_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Age: double (nullable = true)
 |-- Customer_Gender: string (nullable = true)
 |-- Customer_Segment: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_Time: timestamp (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Subcategory: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Discount_Percentage: double (nullable = true)
 |-- Sales_Amount: double (nullable = 

In [57]:
df.show(10)


+--------------+--------+-----------+--------------+------------+---------------+------------------+----------+-------------------+-------------+--------+--------------------+--------------+----------------+-----------+----------+-----------------+----------------+-------------------+--------+----------+-------------------+------------+-----------+------+--------------+------------+---------------+-------------+-----------+-------------+--------------------+--------------+---------------+---------------+----------+
|Transaction_ID|Order_ID|Customer_ID| Customer_Name|Customer_Age|Customer_Gender|  Customer_Segment|Order_Date|         Order_Time|Sales_Channel|Store_ID|          Store_Name|       Country|          Region|       City|Product_ID|     Product_Name|Product_Category|Product_Subcategory|Quantity|Unit_Price|Discount_Percentage|Sales_Amount|Cost_Amount|Profit|Payment_Method|Order_Status|Shipping_Method|Delivery_Days|Return_Flag|Return_Reason|Sales_Representative|Promotion_Code|Cu

In [58]:
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")


Rows: 18045
Columns: 36


In [59]:
from pyspark.sql.functions import col, sum

missing = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing.show()

+--------------+--------+-----------+-------------+------------+---------------+----------------+----------+----------+-------------+--------+----------+-------+------+----+----------+------------+----------------+-------------------+--------+----------+-------------------+------------+-----------+------+--------------+------------+---------------+-------------+-----------+-------------+--------------------+--------------+---------------+---------------+----------+
|Transaction_ID|Order_ID|Customer_ID|Customer_Name|Customer_Age|Customer_Gender|Customer_Segment|Order_Date|Order_Time|Sales_Channel|Store_ID|Store_Name|Country|Region|City|Product_ID|Product_Name|Product_Category|Product_Subcategory|Quantity|Unit_Price|Discount_Percentage|Sales_Amount|Cost_Amount|Profit|Payment_Method|Order_Status|Shipping_Method|Delivery_Days|Return_Flag|Return_Reason|Sales_Representative|Promotion_Code|Customer_Rating|Inventory_Level|Order_Year|
+--------------+--------+-----------+-------------+---------

In [60]:
duplicate_count = (
    df.count() - df.dropDuplicates().count()
)
print("Duplicates:", duplicate_count)

Duplicates: 45


In [61]:
df.dtypes

[('Transaction_ID', 'string'),
 ('Order_ID', 'string'),
 ('Customer_ID', 'string'),
 ('Customer_Name', 'string'),
 ('Customer_Age', 'double'),
 ('Customer_Gender', 'string'),
 ('Customer_Segment', 'string'),
 ('Order_Date', 'date'),
 ('Order_Time', 'timestamp'),
 ('Sales_Channel', 'string'),
 ('Store_ID', 'string'),
 ('Store_Name', 'string'),
 ('Country', 'string'),
 ('Region', 'string'),
 ('City', 'string'),
 ('Product_ID', 'string'),
 ('Product_Name', 'string'),
 ('Product_Category', 'string'),
 ('Product_Subcategory', 'string'),
 ('Quantity', 'int'),
 ('Unit_Price', 'double'),
 ('Discount_Percentage', 'double'),
 ('Sales_Amount', 'double'),
 ('Cost_Amount', 'double'),
 ('Profit', 'double'),
 ('Payment_Method', 'string'),
 ('Order_Status', 'string'),
 ('Shipping_Method', 'string'),
 ('Delivery_Days', 'double'),
 ('Return_Flag', 'string'),
 ('Return_Reason', 'string'),
 ('Sales_Representative', 'string'),
 ('Promotion_Code', 'string'),
 ('Customer_Rating', 'double'),
 ('Inventory_Leve

## Data Cleaning

In [62]:
from pyspark.sql import functions as F
df.groupBy("Customer_Gender").count().orderBy(F.desc("count")).show()
df.select("Customer_Age").summary().show()
df.select(
    F.skewness("Customer_Age").alias("skewness"),
    F.kurtosis("Customer_Age").alias("kurtosis")
).show()

+---------------+-----+
|Customer_Gender|count|
+---------------+-----+
|         Female| 8742|
|           Male| 8438|
|     Non-binary|  543|
|           NULL|  162|
|           male|   77|
|         FEMALE|   75|
|     Non Binary|    8|
+---------------+-----+

+-------+------------------+
|summary|      Customer_Age|
+-------+------------------+
|  count|             17827|
|   mean| 38.91692376731923|
| stddev|10.727356563733347|
|    min|               4.0|
|    25%|              32.0|
|    50%|              39.0|
|    75%|              46.0|
|    max|             112.0|
+-------+------------------+

+-------------------+--------------------+
|           skewness|            kurtosis|
+-------------------+--------------------+
|0.20188018080277645|-0.07245308453676502|
+-------------------+--------------------+



Since the missing data for Customer's Age and Gender are less than 1 percent, we can use mean/median imputation for the nulls.

In [63]:
df = df.dropDuplicates()
total_rows = df.count()

df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) 
    for c in df.columns
]).show()

+--------------+--------+-----------+-------------+------------+------------------+----------------+----------+----------+-------------+--------+----------+-------+------+----+----------+------------+----------------+-------------------+--------+----------+-------------------+------------+-----------+------+--------------+------------+---------------+------------------+-----------+-----------------+--------------------+------------------+------------------+------------------+----------+
|Transaction_ID|Order_ID|Customer_ID|Customer_Name|Customer_Age|   Customer_Gender|Customer_Segment|Order_Date|Order_Time|Sales_Channel|Store_ID|Store_Name|Country|Region|City|Product_ID|Product_Name|Product_Category|Product_Subcategory|Quantity|Unit_Price|Discount_Percentage|Sales_Amount|Cost_Amount|Profit|Payment_Method|Order_Status|Shipping_Method|     Delivery_Days|Return_Flag|    Return_Reason|Sales_Representative|    Promotion_Code|   Customer_Rating|   Inventory_Level|Order_Year|
+--------------+

Handling missing values

In [64]:
# Categorical columns
df = df.fillna({
    "Customer_Name": "Unknown",
    "Customer_Gender": "Unspecified",
    "Return_Reason" : "Not Stated",
    "Promotion_Code": "No Promotion Code",
    "Payment_Method": "Other",
    "Product_Name": "Unlabeled Item"
})

In [65]:
# Numerical columns
from pyspark.sql.functions import median

median_age = df.select(
    median("Customer_Age")
).collect()[0][0]
median_days = df.select(
    median("Delivery_Days")
).collect()[0][0]
median_rating = df.select(
    median("Customer_Rating")
).collect()[0][0]
median_inventory = df.select(
    median("Customer_Rating")
).collect()[0][0]

df = df.fillna({
    "Customer_Age": median_age,
    "Delivery_Days": median_days,
    "Customer_Rating": median_rating,
    "Inventory_Level": median_inventory
})

In [66]:
df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) 
    for c in df.columns
]).show()

+--------------+--------+-----------+-------------+------------+---------------+----------------+----------+----------+-------------+--------+----------+-------+------+----+----------+------------+----------------+-------------------+--------+----------+-------------------+------------+-----------+------+--------------+------------+---------------+-------------+-----------+-------------+--------------------+--------------+---------------+---------------+----------+
|Transaction_ID|Order_ID|Customer_ID|Customer_Name|Customer_Age|Customer_Gender|Customer_Segment|Order_Date|Order_Time|Sales_Channel|Store_ID|Store_Name|Country|Region|City|Product_ID|Product_Name|Product_Category|Product_Subcategory|Quantity|Unit_Price|Discount_Percentage|Sales_Amount|Cost_Amount|Profit|Payment_Method|Order_Status|Shipping_Method|Delivery_Days|Return_Flag|Return_Reason|Sales_Representative|Promotion_Code|Customer_Rating|Inventory_Level|Order_Year|
+--------------+--------+-----------+-------------+---------

In [67]:
df.groupBy("Payment_Method").count().orderBy(F.desc("count")).show()

+--------------+-----+
|Payment_Method|count|
+--------------+-----+
|   Credit Card| 6725|
|    Debit Card| 3888|
|        PayPal| 2793|
| Bank Transfer| 2135|
|     Apple Pay| 1087|
|          Cash| 1038|
|         Other|  144|
|   credit card|   79|
|     DebitCard|   66|
|        paypal|   23|
| Bank transfer|   22|
+--------------+-----+



## Standarlizing Categories

In [68]:
from pyspark.sql.functions import trim, initcap, when

df = df.withColumn(
    "Customer_Gender", 
    initcap(trim(col("Customer_Gender")))
).withColumn(
    "Customer_Gender", 
    when(col("Customer_Gender") == "Non Binary", "Non-binary")
    .otherwise(col("Customer_Gender"))
)
df = df.withColumn(
    "Region",
    initcap(trim(col("Region")))
).withColumn(
    "Region",
    # 2. Map abbreviations and align naming conventions
    when(col("Region") == "Nrw", "North Rhine-Westphalia")
    .when(col("Region") == "North East", "Northeast")
    .otherwise(col("Region"))
)
df = df.withColumn(
    "Region",
    initcap(trim(col("Region")))
) 
df = df.withColumn(
    "City",
    initcap(trim(col("City")))
)
df = df.withColumn(
    "Product_Category",
    initcap(trim(col("Product_Category")))
) 
df = df.withColumn(
    "Order_Status",
    initcap(trim(col("Order_Status")))
)  
df = df.withColumn(
    "Payment_Method",
    F.when(F.lower(F.trim(F.col("Payment_Method"))) == "debitcard", "Debit Card")
     .when(F.lower(F.trim(F.col("Payment_Method"))) == "credit card", "Credit Card")
     .when(F.lower(F.trim(F.col("Payment_Method"))) == "paypal", "PayPal")
     .when(F.lower(F.trim(F.col("Payment_Method"))) == "bank transfer", "Bank Transfer")
     .when(F.lower(F.trim(F.col("Payment_Method"))) == "apple pay", "Apple Pay")
     .when(F.lower(F.trim(F.col("Payment_Method"))) == "cash", "Cash")
     .when(F.lower(F.trim(F.col("Payment_Method"))) == "other", "Other")
     .otherwise(F.initcap(F.trim(F.col("Payment_Method")))) # Fallback auto-capitalization
)

## Feature Engineering

In [69]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, when, round, hour, dayofweek, date_format,
    quarter, month, row_number
)

# 1. Window for customer purchase sequence
cust_window = Window.partitionBy("Customer_ID").orderBy("Order_Date", "Order_Time")

df = (
    df
    # Financial Metrics 
    .withColumn("Profit_Margin_Pct", round((col("Profit") / col("Sales_Amount")) * 100, 2))
    .withColumn("Unit_Cost", when(col("Quantity") > 0, round(col("Cost_Amount") / col("Quantity"), 2)).otherwise(None))
    .withColumn("Discount_Amount", round((col("Unit_Price") * col("Quantity")) - col("Sales_Amount"), 2))
    .withColumn("Has_Discount", when(col("Discount_Percentage") > 0, 1).otherwise(0))
    .withColumn("Is_Profitable", when(col("Profit") > 0, 1).otherwise(0))
    
    # Date & Time Features
    .withColumn("Order_Month", month(col("Order_Date")))
    .withColumn("Order_Quarter", quarter(col("Order_Date")))
    .withColumn("Day_Name", date_format(col("Order_Date"), "EEEE"))
    .withColumn("Is_Weekend", when(dayofweek(col("Order_Date")).isin(1, 7), 1).otherwise(0))
    .withColumn("Order_Hour", hour(col("Order_Time")))
    .withColumn("Day_Part", 
        when(col("Order_Hour").between(6, 11), "Morning")
        .when(col("Order_Hour").between(12, 16), "Afternoon")
        .when(col("Order_Hour").between(17, 21), "Evening")
        .otherwise("Night")
    )
    
    # Customer Demographics & Behavior
    .withColumn("Age_Group",
        when(col("Customer_Age") < 25, "<25")
        .when(col("Customer_Age").between(25, 34), "25-34")
        .when(col("Customer_Age").between(35, 49), "35-49")
        .when(col("Customer_Age") >= 50, "50+")
        .otherwise("Unknown")
    )
    .withColumn("Customer_Order_Sequence", row_number().over(cust_window))
    .withColumn("Is_First_Purchase", when(col("Customer_Order_Sequence") == 1, 1).otherwise(0))
    
    # Fulfillment & Returns
    .withColumn("Is_Returned", when(col("Return_Flag") == "Yes", 1).otherwise(0))
    .withColumn("Delivery_Speed_Category",
        when(col("Delivery_Days") <= 1, "Next Day / Express")
        .when(col("Delivery_Days").between(2, 4), "Standard")
        .when(col("Delivery_Days") >= 5, "Delayed")
        .otherwise("Unknown")
    )
    .withColumn("Stock_Status",
        when(col("Inventory_Level") < 30, "Low Stock").otherwise("Adequate")
    )
    
    # Feedback / Rating
    .withColumn("Is_Rated", when(col("Customer_Rating").isNotNull(), 1).otherwise(0))
    .withColumn("Rating_Class",
        when(col("Customer_Rating") >= 4.0, "Positive")
        .when(col("Customer_Rating") == 3.0, "Neutral")
        .when(col("Customer_Rating") <= 2.0, "Negative")
        .otherwise("No Review")
    )
)


In [70]:
df.write.mode("overwrite").parquet("D:/Portfolio/Retail Sales Dataset/retail-sales-analysis/clean_sales.parquet")


In [71]:
cleaned_df = spark.read.parquet("D:/Portfolio/Retail Sales Dataset/retail-sales-analysis/clean_sales.parquet")
cleaned_df.show() 

+--------------+--------+-----------+----------------+------------+---------------+------------------+----------+-------------------+-------------+--------+--------------------+--------------+--------------------+-----------+----------+-------------------+----------------+-------------------+--------+----------+-------------------+------------+-----------+------+--------------+------------+---------------+-------------+-----------+-------------+--------------------+-----------------+---------------+---------------+----------+-----------------+---------+---------------+------------+-------------+-----------+-------------+---------+----------+----------+---------+---------+-----------------------+-----------------+-----------+-----------------------+------------+--------+------------+
|Transaction_ID|Order_ID|Customer_ID|   Customer_Name|Customer_Age|Customer_Gender|  Customer_Segment|Order_Date|         Order_Time|Sales_Channel|Store_ID|          Store_Name|       Country|              